# 01_bronze_hl7
Loads the legacy HL7 v2 ADT feed into `health.bronze.hl7_messages`, **one row per message**, exactly as received, plus audit columns.

Nothing is parsed or cleaned here; that happens in silver. Keeping the raw text means silver can always be rebuilt.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

files = [f.name for f in dbutils.fs.ls(landing(HL7_VOLUME)) if f.name.endswith(".hl7")]
print(f"{len(files)} HL7 files waiting in landing")
if not files:
    dbutils.notebook.exit("No new HL7 files in landing; nothing to load.")

## Read the files
Each line in a file is one message; segments inside a message are separated by `\r`.
`lineSep = "\n"` is essential: without it Spark also splits on `\r` and breaks every message into separate segment rows.

In [0]:
df = (
    spark.read
        .option("lineSep", "\n")
        .text(landing(HL7_VOLUME) + "*.hl7")
        .filter(F.length("value") > 0)
        .select(
            F.col("value").alias("raw_message"),
            F.sha2("value", 256).alias("message_hash"),          # identical re-sent messages share a hash
            F.col("_metadata.file_name").alias("source_file"),
            F.col("_metadata.file_size").alias("source_file_size"),
            F.current_timestamp().alias("ingested_at"),
        )
)
print("Messages read:", df.count())

In [0]:
# Quick look: show the first message with each segment on its own line
first = df.select("raw_message").first()[0]
print(first.replace("\r", "\n"))

## Write to bronze

In [0]:
(df.write
   .format("delta")
   .mode("append")
   .option("delta.enableChangeDataFeed", "true")
   .saveAsTable(f"{CATALOG}.{BRONZE}.hl7_messages"))

move_to_processed(HL7_VOLUME, files)

## Validate

In [0]:
%sql
-- Expect 3,722 messages from 135 files; duplicates (re-sent messages) are kept on purpose
SELECT COUNT(*)                     AS messages,
       COUNT(DISTINCT source_file)  AS files,
       COUNT(DISTINCT message_hash) AS distinct_messages,
       COUNT(*) - COUNT(DISTINCT message_hash) AS duplicate_messages
FROM health.bronze.hl7_messages